In [5]:
!python3 cli/finance.py -ti AAPL -fi data/aapl.csv

#### Fama-French dataset

[[1]](https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/data_library.html) Dartmouth College, French, K.R (2025) *Current Research Returns* Available at: https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/data_library.html (Accessed: 17 December 2025)

In [ ]:
from pandas import DataFrame, read_csv

path: str = "data/F-F_Research_Data_5_Factors_2x3_daily.csv"
fama_french: DataFrame = read_csv(path)

fama_french.index = fama_french.index.astype("int32")

In [ ]:
from pandas import DataFrame, Series, read_csv, to_datetime

path: str = "data/aapl.csv"
ticker: DataFrame = read_csv(path)
ticker.index = to_datetime(ticker.index, utc = True).strftime("%Y%m%d")
ticker.index = ticker.index.astype("int32")

returns: Series = ticker["Close"].pct_change() * 100
returns.dropna(inplace = True)

In [6]:
from pandas import merge, DataFrame

data: DataFrame = merge(returns, fama_french, how = "inner", left_index = True, right_index = True)
data.to_csv("data/fama_french_data.csv", index = False)

In [22]:
# Metrics declaration
import numpy as np

class Metrics():
  @classmethod
  def mean_squared(cls, error: np.ndarray) -> float:
    m: int = error.shape[0]
    return ((error.T @ error) * (1 / m)).item()

  @classmethod
  def root_mean_squared(cls, error: np.ndarray) -> float:
    return np.sqrt(cls.mean_squared(error))

  @classmethod
  def r_squared(cls, error: np.ndarray, variance: np.ndarray) -> float:
    return (1 - ((error.T @ error) / (variance.T @ variance))).item()

In [20]:
import numpy as np

class Multi_Linear_Regression:
  C: np.ndarray 
  M: np.ndarray 

  def train(self, x_train: np.ndarray, y_train: np.ndarray, lr: float = 1e-5, epoch: int = 1000) -> None:
    m, n = x_train.shape[0], x_train.shape[1]

    self.M: np.ndarray = np.ones((n, 1))
    self.C: np.ndarray = np.ones((1, 1))
    for _ in range(epoch): 
      y_hat: np.ndarray = (x_train @ self.M) + self.C
      error: np.ndarray = y_hat - y_train

      M_der: np.ndarray = (x_train.T @ error) * (2 / m)
      C_der: np.ndarray = (np.ones(m) @ error) * (2 / m)

      self.C: np.ndarray = self.C - (lr * C_der)
      self.M: np.ndarray = self.M - (lr * M_der)
  
  def predict(self, x_test: np.ndarray) -> np.ndarray:
    return (x_test @ self.M) + self.C

In [ ]:
from pandas import DataFrame, read_csv
from sklearn.model_selection import TimeSeriesSplit
import numpy as np

data: DataFrame = read_csv("data/fama_french_data.csv")
x: DataFrame = data.iloc[:, 1:]
y: DataFrame = data[["Close"]]

split: TimeSeriesSplit = TimeSeriesSplit(n_splits = 10)

for fold, (train_index, test_index) in enumerate(split.split(x)):
  x_train: np.ndarray = x.iloc[train_index].values
  x_test: np.ndarray = x.iloc[test_index].values
  y_train: np.ndarray = y.iloc[train_index].values
  y_test: np.ndarray = y.iloc[test_index].values

  model: Multi_Linear_Regression = Multi_Linear_Regression()
  model.train(x_train, y_train, lr = 2e-3,epoch = 100000)  

  y_hat: np.ndarray = model.predict(x_test)

  mean: np.ndarray = np.mean(y_test, dtype = np.ndarray)
  variance: np.ndarray = y_test - mean
  error: np.ndarray = y_hat - y_test

  print(f"Mean squared error: {Metrics.mean_squared(error)}")
  print(f"Root mean squared error: {Metrics.root_mean_squared(error)}")
  print(f"R2 score: {Metrics.r_squared(error, variance)}")

Mean squared error: 5.250939125865871
Root mean squared error: 2.2914927723791476
R2 score: 0.40867528494027927
Mean squared error: 4.940556438066829
Root mean squared error: 2.2227362502255703
R2 score: 0.21363926225471175
Mean squared error: 8.179966618765691
Root mean squared error: 2.860064093471629
R2 score: 0.09010117430139097
Mean squared error: 16.84072156433837
Root mean squared error: 4.10374482203004
R2 score: 0.14436102119003213
Mean squared error: 5.8131743032872425
Root mean squared error: 2.4110525301799717
R2 score: 0.27222269822718426
Mean squared error: 4.901964252430391
Root mean squared error: 2.2140379970611144
R2 score: 0.3635077269970265
Mean squared error: 1.9253329419583134
Root mean squared error: 1.3875636713168564
R2 score: 0.37263846230497455
Mean squared error: 1.526887177538102
Root mean squared error: 1.2356727631286943
R2 score: 0.27818131199141194
Mean squared error: 1.8390754101726177
Root mean squared error: 1.3561251454687424
R2 score: 0.55756861039

In [ ]:
import numpy as np
from pandas import read_csv, DataFrame


data: DataFrame = read_csv("data/fama_french_data.csv")
x: np.ndarray = data.iloc[:, 1:].values
y: np.ndarray = data[["Close"]].values

model: Multi_Linear_Regression = Multi_Linear_Regression()
model.train(x, y) 

[[1.01358858]
 [0.99156975]
 [0.98048263]
 [0.99488291]
 [0.98896402]
 [0.9996953 ]]
[[0.98055003]]
